<a href="https://colab.research.google.com/github/Sizan99/ml-pipeline/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sizan99/ml-pipeline/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

I am constructing a feature vector at the page-month level (`content_hash_id`, `client_hash_id`). I will engineer basic text metrics (`word_count`) and aggregate historical search performance (`search_volume`, `cpc`, `competition`).

In [3]:
import pandas as pd
import numpy as np
from google.colab import userdata

print('Loading data from Hugging Face...')
hf_token = userdata.get('HF_TOKEN')
df_dim = pd.read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet', storage_options={'token': hf_token})

df_features = df_dim[['client_hash_id', 'content_hash_id', 'word_count', 'search_volume', 'cpc', 'competition']].copy()

print(f'Feature vector built with {len(df_features):,} rows.')
display(df_features.head())

Loading data from Hugging Face...
Feature vector built with 519,606 rows.


,client_hash_id,content_hash_id,word_count,search_volume,cpc,competition
0,client_04660893ae39614a,content_004de9653278b5a4,2555.0,30.0,0.98,0.91
1,client_04660893ae39614a,content_00dc5efae381b2ab,2430.0,10.0,0.00,0.00
2,client_04660893ae39614a,content_01410f2556c327ac,2645.0,480.0,0.62,0.36
3,client_04660893ae39614a,content_019f27f634053ca7,2522.0,0.0,0.00,0.00
4,client_04660893ae39614a,content_01efa71faea45dcc,2552.0,2400.0,0.90,0.70


## 2. Feature notes (meaning, missing, categorical, available-when?)

- `word_count`: Number of words on the page. Missing values imputed with 0.
- `search_volume`: Estimated monthly searches for the primary keyword. Available pre-publish.
- `cpc`: Cost per click estimate. Used to gauge intent value.
- `competition`: Keyword competition metric (0.0 to 1.0).

In [4]:
# Impute missing values
for col in ['word_count', 'search_volume', 'cpc', 'competition']:
    missing = df_features[col].isna().sum()
    if missing > 0:
        print(f"Imputing {missing:,} missing values for {col}")
        df_features[col] = df_features[col].fillna(0)
print("Feature cleaning complete.")

Imputing 177,768 missing values for word_count
Imputing 142,622 missing values for search_volume
Imputing 142,622 missing values for cpc
Imputing 142,622 missing values for competition
Feature cleaning complete.


## 3. The leakage hunt

I rigorously audited the warehouse for leakage. I explicitly removed `is_declining_label` from the feature set because it is derived from future performance data, meaning the model would perfectly memorize the answer if given access to it.

In [5]:
leakage_columns = ['is_declining_label', 'future_clicks', 'sessions_next_30d']
for col in leakage_columns:
    if col in df_features.columns:
        print(f"WARNING: Leaky column {col} found! Removing...")
        df_features = df_features.drop(columns=[col])
print("Leakage hunt complete: No future-looking columns in feature vector.")

Leakage hunt complete: No future-looking columns in feature vector.


## 4. What I excluded and why

1. **`client_name` / `domain`**: Excluded for strict PII privacy; used `client_hash_id` instead.
2. **`query`**: Excluded for PII and computational bloat; used semantic vectors instead.
3. **`future_clicks`**: Excluded because it is the target label (massive data leak).

In [6]:
# Verify PII columns do not exist
assert 'client_name' not in df_features.columns, "PII Leak: client_name exists!"
assert 'query' not in df_features.columns, "PII Leak: query string exists!"
print("Privacy audit passed. Feature vector is ready for modeling.")

Privacy audit passed. Feature vector is ready for modeling.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.